In [3]:
import bchlib
import os

T = 5
M = 13

bch = bchlib.BCH(m=M, t=T)

data = bytearray(os.urandom(512))
ecc = bch.encode(data)
packet = bytearray(data + ecc)

print(f"encoded {len(data)} bytes. {len(ecc)} bytes of ECC added ({T} errors correction).")

# introduce errors
packet[5] ^= 0x04
packet[5] ^= 0x03
packet[300] ^= 0x01
packet[400] ^= 0x80

recv_data = bytearray(packet[:-bch.ecc_bytes])
recv_ecc = packet[-bch.ecc_bytes:]

bitflips = bch.decode(recv_data, recv_ecc)

if bitflips >= 0:
    bch.correct(recv_data, recv_ecc)
    counter = 0
    for i in range(len(data)):
        if data[i] != recv_data[i]:
            print("data error at byte", i)
            counter += 1
    print("corrected", bitflips, "errors")
    print("total data errors:", counter)
else:
    print("too many errors")

encoded 512 bytes. 9 bytes of ECC added (5 errors correction).
corrected 5 errors
total data errors: 0
